# step2 자리 통제 — "위반을 더 본다"가 진짜인가

**어느 스텝:** step2 보강 · **RQ:** RQ2(관측) · **무엇을 확인:** 표기 격차가 **표기 때문인가
자리 때문인가**

## 왜 필요한가 — 교란을 발견했다

토큰별 어텐션을 처음 그려 봤더니(`scripts/step2_token_bars.py`), 어텐션이
**프롬프트에 나온 순서를 따라 떨어진다.** 그런데 **42묶음이 전부 같은 배치**다:

```
   자리   1     2     3     4     5     6     7     8     9    10    11    12
   표기   S     C     C     S     S     S     S     C     C     C     C     S
   어텐션 .0515 .0404 .0283 .0265 .0222 .0187 .0150 .0182 .0126 .0140 .0124 .0209
          ↑ 1번이 나머지 평균의 2.5배
```

묶음 번호는 **어떤 이름을 쓸지**만 바꾸고, 표기 배치는 **시드가 정한다**
(`prompt.py:151` — `random.Random(condition.seed).shuffle(notations)`).
시드가 전부 42라 **42묶음 모두 위반이 1번 자리**다.

**1번 자리를 빼고 이름당 평균으로 다시 재면 격차가 사라진다:**

| 모델 | 전체 차이(위반−준수) | **1번 뺀 차이** |
|---|---|---|
| Qwen | +0.0290 | **−0.0003** |
| DeepSeek | +0.0197 | **−0.0005** |
| Llama | +0.0075 | **−0.0001** |
| StableCode | +0.0244 | +0.0042 |

**3모델에서 0으로 사라지고 부호까지 뒤집힌다.** 즉 지금 데이터로는
**"위반을 더 본다"와 "1번 자리를 더 본다"를 가를 수 없다** — 모든 묶음에서 둘이 100% 겹친다.

## 어떻게 가르나 — 정확한 반대 배치

시드 67이 시드 42의 **정확한 반대**다(1,014개 시드를 훑어 찾았다).

```
   시드 42 :  S C C S S S S C C C C S
   시드 67 :  C S S C C C C S S S S C     ← 모든 자리가 뒤집힌다
```

두 시드를 합치면 **모든 자리에서 위반이 정확히 절반**이 된다. 무작위 시드를 여러 개
뽑는 것과 다르다 — 시드 8개를 써도 자리별 편중이 0.38 남는데, 이 방법은 **정확히 0**이다.

### 왜 이게 검정이 되나

```
   자리 효과만 있다면 (귀무가설)
     시드42 위반 = 자리{1,4,5,6,7,12}
     시드67 위반 = 자리{2,3,8,9,10,11}
     ─────────────────────────────────
     두 시드 평균 = 12자리 전체 ÷ 2  =  준수 평균      →  차이 0

   위반이라서 더 보는 효과 Δ가 있다면
     시드42 위반 = 자리{1,4,5,6,7,12} + Δ×6
     시드67 위반 = 자리{2,3,8,9,10,11} + Δ×6
     ─────────────────────────────────────
     평균 − 준수 평균 = Δ×6        ← 자리는 완전 소거, Δ만 남는다
```

**자리를 흐리는 게 아니라 수학적으로 지운다.**

## 무엇을 바꾸나

**시드 하나뿐이다.** 프롬프트·묶음·모델·측정법 전부 그대로다.

| | 기존 | 이번 |
|---|---|---|
| 시드 | 42 | **67** |
| 그 외 전부 | — | **동일** |

**조건 수:** 4모델 × 42묶음 = **168개**. `observe` 모드라 조건당 forward 1회 —
이 연구에서 **가장 가벼운 실험**이다.

## 결과가 어느 쪽으로 나오든 무슨 뜻인지 (미리 밝힌다)

| 두 시드 합친 격차 | 뜻 | 문서에 어떻게 |
|---|---|---|
| **0 근처** (신뢰구간이 0을 문다) | **전부 자리 효과였다** | step2 §4-(1) "위반을 더 본다"를 **철회**한다. "관측으로는 못 가른다 → 그래서 인과(step3)가 필요하다"로 바꾼다 |
| **양수로 남는다** | **진짜 위반 효과다.** 그 값이 Δ×6 | §4-(1)을 **자리 통제 후 값으로 다시 쓴다.** 기존 값은 상한이었다고 적는다 |
| 모델마다 갈린다 | 모델 의존 | 갈리는 대로 적는다 |

**예상:** 1번 자리를 뺐을 때 3모델에서 0이었으므로 **0에 가깝게 나올 것으로 본다.**
다르게 나오면 조건을 바꾸지 않고 **그대로 기록한다**(CLAUDE.md §4).

> **어느 쪽이 나와도 step3(인과)는 안 흔들린다.** step3은 **같은 6자리**를 Value로 덮었을 때와
> Key로 덮었을 때를 비교하므로 자리가 양쪽에 똑같이 들어가 상쇄된다.
> step5는 지침 지시어 한 단어가 대상이라 선행 코드 배치와 무관하고, 실제로 지침 방향을
> 뒤집어(=배치가 통째로 뒤집힘) 봐도 봉우리 층이 4모델 다 그대로다.

## 부하

T4에서 **모델당 5~10분**. DeepSeek-6.7B만 메모리를 조금 쓴다.

In [ ]:
# ② 환경 · 시드
!pip install -q -r requirements.txt
import random, numpy as np, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (매우 느림)')
SEED = 67          # ★ 이 노트북의 전부. 기존 실행분은 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('이번 시드:', SEED)

In [ ]:
# ③ 저장소 · 브랜치
import os
if not os.path.isdir('HCLT_2026'):
    !git clone https://github.com/deanjs/HCLT_2026.git
%cd HCLT_2026
BRANCH = 'integration/step1-5'
!git fetch --quiet origin $BRANCH
!git checkout $BRANCH
!git pull --quiet origin $BRANCH
!pip install -e . -q
import sys; sys.path.insert(0, 'src')
print('브랜치:', BRANCH)

In [ ]:
# ④ 조건 설정 — 시드만 67, 나머지는 기존과 동일
from harness.conditions import (Condition, ModelSpec, PrecedingCode, Instruction,
                                Composition, InstructionForm, Notation, Intervention)

MODELS = [
    ModelSpec(name='Qwen/Qwen2.5-Coder-3B-Instruct',           family='qwen',      dtype='float16'),
    ModelSpec(name='deepseek-ai/deepseek-coder-6.7b-instruct',  family='deepseek',  dtype='float16'),
    ModelSpec(name='unsloth/Llama-3.2-3B-Instruct',             family='llama',     dtype='float16'),
    ModelSpec(name='stabilityai/stable-code-instruct-3b',       family='stability', dtype='float16'),
]

PICK = 0                      # 0=qwen 1=deepseek 2=llama 3=stable
MODEL = MODELS[PICK]
BLOCKS = list(range(42))      # 기존과 같은 42묶음

conditions = [
    Condition(model=MODEL,
              preceding=PrecedingCode(n_compliant=6, n_functions=12,
                                      composition=Composition.POOL, pool_block=b),
              instruction=Instruction(form=InstructionForm.POSITIVE,
                                      target_notation=Notation.CAMEL),
              intervention=Intervention(), seed=SEED)
    for b in BLOCKS
]
print(f'{MODEL.family}: 조건 {len(conditions)}개')
assert len({c.slug() for c in conditions}) == len(conditions)

# ★ 배치가 정말 뒤집혔는지 눈으로 확인한다 — 이게 이 실험의 전부다
import re
from harness.prompt import build_preceding_code
for s, tag in ((42, '기존'), (SEED, '이번')):
    c = Condition(model=MODEL,
                  preceding=PrecedingCode(n_compliant=6, n_functions=12,
                                          composition=Composition.POOL, pool_block=0),
                  instruction=Instruction(form=InstructionForm.POSITIVE,
                                          target_notation=Notation.CAMEL),
                  intervention=Intervention(), seed=s)
    names = re.findall(r'def (\w+)\(', build_preceding_code(c))
    print(f'  시드 {s:>3} ({tag}): ' + ' '.join('S' if '_' in n else 'C' for n in names))
print('  → 모든 자리가 뒤집혀 있어야 정상 (S↔C)')

In [ ]:
# ⑤ 실행 — 이미 있으면 건너뛴다(재개)
from harness import run, ResultRecord, save_result, result_path
from harness.model import load_model
import numpy as np

STEP = 'step2_code-observe'     # 기존과 같은 폴더. 슬러그의 __s67이 구분한다
todo = [c for c in conditions if not result_path(c, step=STEP).exists()]
print(f'[{MODEL.family}] 전체 {len(conditions)}개 중 남은 조건 {len(todo)}개')

if todo:
    handle = load_model(MODEL)
    seen = []
    for i, c in enumerate(todo, 1):
        out = run(c, handle=handle, mode='observe')
        save_result(ResultRecord(condition=out.condition, metrics=out.metrics,
                                 step=STEP, rq='RQ2'))
        ex = out.metrics.extra
        cnt = ex['span_token_counts']
        pk = max(out.metrics.per_layer,
                 key=lambda L: out.metrics.per_layer[L].get('code_snake__attention_weight', 0))
        v = out.metrics.per_layer[pk]
        seen.append((v['code_snake__attention_weight'] / cnt['code_snake'],
                     v['code_camel__attention_weight'] / cnt['code_camel']))
        del out
        if i % 10 == 0 or i == len(todo):
            s = np.mean([a for a, _ in seen]); c2 = np.mean([b for _, b in seen])
            print(f'  [{i}/{len(todo)}] 봉우리 층 토큰당 — 위반 {s:.5f} · 준수 {c2:.5f} · 차이 {s-c2:+.5f}')
    del handle
    import torch, gc; gc.collect(); torch.cuda.empty_cache()
print('완료')

In [ ]:
# ⑥ 요약 — 두 시드를 합치면 격차가 남는가 (이 실험의 답)
import json, glob
import numpy as np
from collections import defaultdict

rows = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))
for f in glob.glob('results/step2_code-observe/*.json'):
    r = json.load(open(f, encoding='utf-8'))
    if r['condition']['model']['family'] != MODEL.family:
        continue
    sd = r['condition'].get('seed')
    cnt = r['metrics']['extra']['span_token_counts']
    for L, v in r['metrics']['per_layer'].items():
        for sp in ('code_snake', 'code_camel'):
            x = v.get(sp + '__attention_weight')
            if x is not None:
                rows[sd][int(L)][sp].append(x / cnt[sp])      # 토큰당

have = sorted(k for k in rows if k is not None)
print(f'[{MODEL.family}] 있는 시드: {have}')
if len(have) < 2:
    print('→ 시드가 하나뿐이다. 42와 67 둘 다 있어야 자리가 소거된다.')
else:
    layers = sorted(rows[have[0]])
    print(f"\n{'층':>5}" + ''.join(f'{f"s{s} 차이":>12}' for s in have) + f"{'두 시드 평균':>14}")
    best = None
    for L in layers:
        diffs = [np.mean(rows[s][L]['code_snake']) - np.mean(rows[s][L]['code_camel'])
                 for s in have]
        avg = float(np.mean(diffs))
        if best is None or abs(avg) > abs(best[1]):
            best = (L, avg)
        if L % 5 == 0 or abs(avg) > 0.0005:
            print(f'{L:>5}' + ''.join(f'{d:>12.5f}' for d in diffs) + f'{avg:>14.5f}')
    print(f'\n절대값이 가장 큰 층: L{best[0]}  두 시드 평균 차이 {best[1]:+.5f}')
    print('\n읽는 법')
    print('  두 시드 평균 차이가 0 근처면 → 격차는 **자리** 때문이었다. §4-(1)을 철회한다.')
    print('  양수로 남으면          → 그게 자리를 지운 **진짜 위반 효과**다.')

In [ ]:
# ⑦ 이번에 만든 것만 zip으로 내려받기
import shutil, os, glob
from harness import result_path

STEP = 'step2_code-observe'
made = [result_path(c, step=STEP) for c in conditions if result_path(c, step=STEP).exists()]
print(f'폴더 전체 {len(glob.glob(f"results/{STEP}/*.json"))}개 / 이번({MODEL.family}, 시드{SEED}) {len(made)}개')
# ⚠️ 폴더에는 기존 시드 42 결과가 이미 들어 있다. 통째로 묶으면 그것까지 딸려 나가고,
#    다시 풀어 커밋하면 불변이어야 할 결과를 덮어쓴다(CLAUDE.md §6).
if made:
    stage = f'upload_step2_s{SEED}_{MODEL.family}'
    shutil.rmtree(stage, ignore_errors=True); os.makedirs(stage)
    for f in made:
        shutil.copy(f, stage)
    path = shutil.make_archive(stage, 'zip', stage)
    print(f'\n{path} ({os.path.getsize(path)/1e6:.1f}MB, {len(made)}개)')
    try:
        from google.colab import files; files.download(path)
    except Exception as e:
        print('직접 받을 것:', e)
    print('\n받은 zip은 results/step2_code-observe/ 에 **풀어 넣기만** 한다(같은 이름 없어야 정상).')
    print('네 모델을 다 모은 뒤:  python scripts/step2_position_control.py')